# Практическая работа №6. Классификация текстов

ФИО: ____________________


# Практическая работа №6. Классификация текстов

## Предобработка

Текст приводится к нижнему регистру, удаляются пунктуация, цифры и часть стоп-слов. После этого строятся признаки Bag of Words и TF-IDF.

## Сравнение векторизации

| Метод | Accuracy | F1 | Размерность |
| --- | ---: | ---: | ---: |
| CountVectorizer | 0.500 | 0.571 | 72 |
| TfidfVectorizer | 0.500 | 0.571 | 72 |

## Анализ ошибок

Количество ошибок TF-IDF на test: 3. На маленьком корпусе часть ошибок связана с тем, что отдельные слова встречаются редко, а модель Naive Bayes не понимает контекст и иронию.


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

def clean_text(value):
    allowed = [ch if ch.isalpha() or ch.isspace() else " " for ch in value.lower()]
    stop_words = {"и", "в", "на", "с", "по", "как", "но", "не", "что"}
    return " ".join(word for word in "".join(allowed).split() if word not in stop_words)

df = pd.read_csv("../data/text_reviews.csv")
df["clean_text"] = df["text"].map(clean_text)
X_train, X_test, y_train, y_test = train_test_split(df["clean_text"], df["label"], test_size=0.3, random_state=42, stratify=df["label"])

for name, vectorizer in [("bow", CountVectorizer()), ("tfidf", TfidfVectorizer(min_df=1, max_df=0.95))]:
    model = Pipeline([("vectorizer", vectorizer), ("classifier", MultinomialNB())])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(name, "accuracy=", accuracy_score(y_test, pred), "f1=", f1_score(y_test, pred))
    print("features=", len(model.named_steps["vectorizer"].vocabulary_))
